# Final Hybrid Evaluation

Compare ALS with LightFM, then test whether a content-ranked warm-tail item can safely replace the last ALS recommendation. The live application remains ALS-only; this notebook records the research evaluation.


## Evaluation setup

Positive interaction: rating ≥ 4.0. Validation targets begin on 1 January 2016 (UTC). Tail items have fewer than 20 positive training interactions; warm-tail items have 1–19. All validation models use the same users and full catalogue.


In [1]:
# Setup and dependency checks
import os
import sys
import time
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.preprocessing import normalize

try:
    import lightfm
    from lightfm import LightFM
except ImportError as exc:
    raise ImportError(
        "LightFM is not installed in this notebook's Python environment. "
        "In the project's active environment run: python -m pip install lightfm. "
        "If it fails to build on Windows, send the complete error output."
    ) from exc

# Find the project root, then use the same notebooks/ working directory convention
# used by the existing project notebooks and scripts/recutils.py.
_START = Path.cwd().resolve()
_PROJECT_ROOT = None
for _p in [_START, *_START.parents]:
    if (_p / "scripts" / "recutils.py").exists() and (_p / "data").exists():
        _PROJECT_ROOT = _p
        break
if _PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find project root containing scripts/recutils.py and data/. "
        "Place this notebook in the project's notebooks/ folder and reopen it there."
    )
_NOTEBOOK_DIR = _PROJECT_ROOT / "notebooks"
if _NOTEBOOK_DIR.exists() and Path.cwd().resolve() != _NOTEBOOK_DIR.resolve():
    os.chdir(_NOTEBOOK_DIR)
sys.path.insert(0, str(_PROJECT_ROOT / "scripts"))

NAME, K, REL, SEED = "sample_10pct", 10, 4.0, 42
CUTOFF_TS = int(pd.Timestamp("2016-01-01 00:00:00", tz="UTC").timestamp())
RETENTION_FRACTION = 0.98
N_BOOT = 1000
N_TIE_SEEDS = 10
SCORE_BATCH_SIZE = 64
NUM_THREADS = 1  # reproducibility first; increase only if runtime requires it

# First controlled LightFM configuration. Keep this fixed across the three models.
LIGHTFM_COMPONENTS = 32
LIGHTFM_EPOCHS = 15
LIGHTFM_LEARNING_RATE = 0.05
LIGHTFM_ITEM_ALPHA = 1e-6
LIGHTFM_USER_ALPHA = 1e-6
LIGHTFM_MAX_SAMPLED = 10

print("Project root:", _PROJECT_ROOT)
print("LightFM version:", getattr(lightfm, "__version__", "version unavailable"))
print("LightFM config:", {
    "loss": "warp",
    "no_components": LIGHTFM_COMPONENTS,
    "epochs": LIGHTFM_EPOCHS,
    "learning_rate": LIGHTFM_LEARNING_RATE,
    "item_alpha": LIGHTFM_ITEM_ALPHA,
    "user_alpha": LIGHTFM_USER_ALPHA,
    "max_sampled": LIGHTFM_MAX_SAMPLED,
    "num_threads": NUM_THREADS,
})


Project root: /mnt/c/Users/Lenovo/Documents/movie-recommender
LightFM version: 1.17
LightFM config: {'loss': 'warp', 'no_components': 32, 'epochs': 15, 'learning_rate': 0.05, 'item_alpha': 1e-06, 'user_alpha': 1e-06, 'max_sampled': 10, 'num_threads': 1}


## Load the split and frozen content features

Only train and validation are loaded here. The test file is read only in the final evaluation section.


In [2]:
DATA_DIR = _PROJECT_ROOT / "data" / "processed" / NAME
train = pd.read_parquet(DATA_DIR / "train.parquet")
val = pd.read_parquet(DATA_DIR / "val.parquet")
item_map = pd.read_parquet(DATA_DIR / "item_map.parquet").sort_values("i").reset_index(drop=True)
user_map = pd.read_parquet(DATA_DIR / "user_map.parquet").sort_values("u").reset_index(drop=True)
n_users = int(user_map["u"].max()) + 1
n_items = int(item_map["i"].max()) + 1

feature_dir = _PROJECT_ROOT / "data" / "interim" / "hybrid_features"
v1_path = feature_dir / "content_v1.npz"
if not v1_path.exists():
    raise FileNotFoundError(
        f"Missing {v1_path}. Run 04_content_based.ipynb first."
    )
X_v1 = sp.load_npz(v1_path).tocsr().astype(np.float32)

assert len(train) and len(val)
assert item_map["i"].is_unique and item_map["movieId"].is_unique
assert user_map["u"].is_unique and user_map["userId"].is_unique
assert np.array_equal(item_map["i"].to_numpy(), np.arange(n_items))
assert X_v1.shape == (n_items, 31)
assert np.isfinite(X_v1.data).all()

# Exclude every training interaction, including ratings below 4.0.
seen_train = sp.csr_matrix(
    (
        np.ones(len(train), dtype=np.float32),
        (train["u"].to_numpy(dtype=np.int64), train["i"].to_numpy(dtype=np.int64)),
    ),
    shape=(n_users, n_items),
)
seen_train.sum_duplicates()
seen_train.data[:] = 1.0

train_positive = (
    train.loc[train["rating"] >= REL, ["u", "i"]]
    .drop_duplicates(["u", "i"])
)
n_pos = np.bincount(train_positive["i"].to_numpy(dtype=np.int64), minlength=n_items)
tail_mask = n_pos < 20
strict_cold_mask = n_pos == 0
warm_tail_mask = (n_pos >= 1) & (n_pos < 20)
item_has_positive_history = n_pos > 0
popularity_scores = np.bincount(
    train["i"].to_numpy(dtype=np.int64), minlength=n_items
).astype(np.float64)
u_has_pos = (
    np.bincount(train_positive["u"].to_numpy(dtype=np.int64), minlength=n_users) > 0
)

val_ts = pd.to_numeric(val["timestamp"], errors="raise")
val_window = val.loc[
    (val["rating"] >= REL) & (val_ts >= CUTOFF_TS),
    ["u", "i", "timestamp"],
].drop_duplicates(["u", "i"]).copy()

def relevant_dict(frame):
    return {
        int(user): set(group["i"].astype(int).tolist())
        for user, group in frame.groupby("u", sort=False)
    }

item_idx_val = val_window["i"].to_numpy(dtype=np.int64)
relevant_by_scope = {
    "overall": relevant_dict(val_window),
    "tail": relevant_dict(val_window.loc[tail_mask[item_idx_val]]),
    "strict_cold": relevant_dict(val_window.loc[strict_cold_mask[item_idx_val]]),
    "warm_tail": relevant_dict(val_window.loc[warm_tail_mask[item_idx_val]]),
}

# Explicit alias used by the selective reranking and paired-bootstrap sections.
overall_relevant = relevant_by_scope["overall"]
scope_users = {
    scope: np.asarray(sorted(items), dtype=np.int64)
    for scope, items in relevant_by_scope.items()
}
eval_users = scope_users["overall"]

target_u = val_window["u"].to_numpy(dtype=np.int64)
target_i = val_window["i"].to_numpy(dtype=np.int64)
assert not np.asarray(seen_train[target_u, target_i]).ravel().any()
assert X_v1.shape[0] == n_items

print(f"{n_users:,} users | {n_items:,} movies")
print(f"Train: {len(train):,} | Validation: {len(val):,}")
print(f"Post-2015 validation targets: {len(val_window):,} across {len(eval_users):,} users")
print(f"Warm-tail movies: {int(warm_tail_mask.sum()):,} | Strict-cold: {int(strict_cold_mask.sum()):,}")
print("Item alignment and train/validation checks passed. Test data has not been loaded.")


20,093 users | 17,241 movies
Train: 2,919,680 | Validation: 137,197
Post-2015 validation targets: 25,406 across 5,661 users
Warm-tail movies: 10,502 | Strict-cold: 1,095
Item alignment and train/validation checks passed. Test data has not been loaded.


## Build interactions and item features

LightFM is trained on positive interactions. The two item-feature choices are identity-only and identity plus the frozen genre/decade metadata.


In [3]:
# Positive training interactions for LightFM.
interactions = sp.coo_matrix(
    (
        np.ones(len(train_positive), dtype=np.float32),
        (
            train_positive["u"].to_numpy(dtype=np.int32),
            train_positive["i"].to_numpy(dtype=np.int32),
        ),
    ),
    shape=(n_users, n_items),
    dtype=np.float32,
)
interactions.sum_duplicates()
interactions.eliminate_zeros()

# LightFM can use identity features alone or identity plus metadata.
I_items = sp.eye(n_items, format="csr", dtype=np.float32)

def make_identity_plus_metadata_features(X_side, metadata_mass=0.5):
    X_side = normalize(
        X_side.tocsr().astype(np.float32),
        norm="l1", axis=1, copy=True
    ).tocsr()
    has_metadata = np.asarray(X_side.getnnz(axis=1) > 0).ravel()
    identity_weight = np.where(has_metadata, 1.0 - metadata_mass, 1.0)
    identity_block = sp.diags(identity_weight.astype(np.float32)) @ I_items
    metadata_block = X_side.multiply(metadata_mass)
    features = sp.hstack(
        [identity_block, metadata_block], format="csr", dtype=np.float32
    )
    assert np.allclose(np.asarray(features.sum(axis=1)).ravel(), 1.0, atol=1e-5)
    return features

item_features_identity = I_items
item_features_v1 = make_identity_plus_metadata_features(X_v1, metadata_mass=0.5)
item_feature_sets = {
    "LightFM_identity": item_features_identity,
    "LightFM_content_v1": item_features_v1,
}

print("Positive interaction matrix:", interactions.shape, "nnz=", f"{interactions.nnz:,}")
for name, matrix in item_feature_sets.items():
    print(f"{name}: {matrix.shape}, {matrix.nnz:,} nonzero values")


Positive interaction matrix: (20093, 17241) nnz= 1,445,945
LightFM_identity: (17241, 17241), 17,241 nonzero values
LightFM_content_v1: (17241, 17272), 71,200 nonzero values


## Fit the ALS reference

Use the selected implicit ALS configuration from the collaborative-filtering comparison.


In [4]:
# Train-only implicit ALS reference, copied from the validated project setup.
def _implicit_half_step(M, Y, lam):
    """One ALS half-step; M stores confidence-minus-one for observed positives."""
    n_rows = M.shape[0]
    k = Y.shape[1]
    YtY = Y.T @ Y
    reg = lam * np.eye(k)
    out = np.zeros((n_rows, k), dtype=np.float64)
    indptr, indices, data = M.indptr, M.indices, M.data

    for row in range(n_rows):
        start, end = indptr[row], indptr[row + 1]
        if start == end:
            continue
        factors = Y[indices[start:end]]
        confidence_minus_one = data[start:end]
        A = YtY + (factors.T * confidence_minus_one) @ factors + reg
        b = factors.T @ (confidence_minus_one + 1.0)
        out[row] = np.linalg.solve(A, b)
    return out


def fit_implicit_als_train_only(df, k=64, alpha=5.0, lam=80.0, n_iter=10, seed=SEED):
    d = df.loc[df["rating"] >= REL, ["u", "i"]].drop_duplicates(["u", "i"])
    C = sp.csr_matrix(
        (
            np.full(len(d), float(alpha), dtype=np.float64),
            (d["u"].to_numpy(dtype=np.int64), d["i"].to_numpy(dtype=np.int64)),
        ),
        shape=(n_users, n_items),
    )
    C.sum_duplicates()
    C.eliminate_zeros()
    Ct = C.T.tocsr()
    rng = np.random.default_rng(seed)
    Q = rng.normal(0, 0.1, size=(n_items, k))
    P = np.zeros((n_users, k), dtype=np.float64)
    start_time = time.time()

    for iteration in range(1, n_iter + 1):
        P = _implicit_half_step(C, Q, lam)
        Q = _implicit_half_step(Ct, P, lam)
        if iteration == 1 or iteration % 2 == 0 or iteration == n_iter:
            print(f"ALS iteration {iteration:02d}/{n_iter} | elapsed {time.time() - start_time:.1f}s")
    return {"P": P.astype(np.float32), "Q": Q.astype(np.float32)}

als_model = fit_implicit_als_train_only(
    train, k=64, alpha=5.0, lam=80.0, n_iter=10, seed=SEED
)
P_als, Q_als = als_model["P"], als_model["Q"]
item_has_als_factors = np.abs(Q_als).sum(axis=1) > 0
assert P_als.shape == (n_users, 64) and Q_als.shape == (n_items, 64)
assert np.isfinite(P_als).all() and np.isfinite(Q_als).all()
assert int((~item_has_als_factors).sum()) == int(strict_cold_mask.sum())

print("ALS refit complete.")
print("Users without positive history:", int((~u_has_pos).sum()))
print("Items without learned ALS factors:", int((~item_has_als_factors).sum()))

ALS iteration 01/10 | elapsed 11.0s
ALS iteration 02/10 | elapsed 17.8s
ALS iteration 04/10 | elapsed 27.1s
ALS iteration 06/10 | elapsed 39.1s
ALS iteration 08/10 | elapsed 48.2s
ALS iteration 10/10 | elapsed 58.9s
ALS refit complete.
Users without positive history: 34
Items without learned ALS factors: 1095


## Fit LightFM candidates

Both candidates use WARP and the same fixed training settings.


In [5]:
def fit_lightfm_candidate(name, item_features):
    start_time = time.time()
    model = LightFM(
        no_components=LIGHTFM_COMPONENTS,
        loss="warp",
        learning_rate=LIGHTFM_LEARNING_RATE,
        item_alpha=LIGHTFM_ITEM_ALPHA,
        user_alpha=LIGHTFM_USER_ALPHA,
        max_sampled=LIGHTFM_MAX_SAMPLED,
        random_state=SEED,
    )
    model.fit(
        interactions,
        item_features=item_features,
        epochs=LIGHTFM_EPOCHS,
        num_threads=NUM_THREADS,
        verbose=True,
    )
    print(f"{name} fit complete in {(time.time() - start_time) / 60:.1f} min")
    print("User embeddings:", model.user_embeddings.shape)
    print("Item feature embeddings:", model.item_embeddings.shape)
    return model

lightfm_models = {}

In [6]:
# 6a · Identity-only LightFM
lightfm_models["LightFM_identity"] = fit_lightfm_candidate(
    "LightFM_identity",
    item_features_identity,
)

Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
LightFM_identity fit complete in 0.4 min
User embeddings: (20093, 32)
Item feature embeddings: (17241, 32)


In [7]:
# 6b · LightFM with the frozen content-v1 metadata (genres + release decade)
lightfm_models["LightFM_content_v1"] = fit_lightfm_candidate(
    "LightFM_content_v1",
    item_features_v1,
)

Epoch 0


Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
LightFM_content_v1 fit complete in 0.7 min
User embeddings: (20093, 32)
Item feature embeddings: (17272, 32)


## Score and rank the catalogue

Use the full-catalogue scores and the same seen-item exclusions and deterministic tie-breaking rule for every model.


In [8]:
def make_lightfm_scorer(model, item_features):
    # These products are small dense matrices: (n_items, n_components) plus one bias per item.
    item_vectors = np.asarray(item_features @ model.item_embeddings, dtype=np.float32)
    item_bias = np.asarray(item_features @ model.item_biases).reshape(-1).astype(np.float32)
    user_vectors = np.asarray(model.user_embeddings, dtype=np.float32)
    user_bias = np.asarray(model.user_biases, dtype=np.float32)

    assert item_vectors.shape == (n_items, LIGHTFM_COMPONENTS)
    assert user_vectors.shape == (n_users, LIGHTFM_COMPONENTS)
    assert item_bias.shape == (n_items,) and user_bias.shape == (n_users,)

    def score_batch(user_batch):
        ub = np.asarray(user_batch, dtype=np.int32)
        scores = user_vectors[ub] @ item_vectors.T
        scores += user_bias[ub, None]
        scores += item_bias[None, :]
        return scores.astype(np.float32, copy=False)

    return score_batch


def make_als_scorer(P, Q):
    def score_batch(user_batch):
        ub = np.asarray(user_batch, dtype=np.int64)
        return (P[ub] @ Q.T).astype(np.float32, copy=False)
    return score_batch

score_functions = {
    "ALS_only": make_als_scorer(P_als, Q_als),
}
for model_name, model in lightfm_models.items():
    score_functions[model_name] = make_lightfm_scorer(
        model, item_feature_sets[model_name]
    )

# Smoke-test direct matrix scoring against LightFM's official predict() on each fitted model.
smoke_user = int(eval_users[0])
smoke_items = np.asarray([0, min(1, n_items - 1), min(17, n_items - 1)], dtype=np.int32)
for model_name, model in lightfm_models.items():
    feature_matrix = item_feature_sets[model_name]
    official = model.predict(
        user_ids=np.full(len(smoke_items), smoke_user, dtype=np.int32),
        item_ids=smoke_items,
        item_features=feature_matrix,
        num_threads=NUM_THREADS,
    )
    direct = score_functions[model_name](np.asarray([smoke_user]))[0, smoke_items]
    assert np.allclose(official, direct, rtol=1e-4, atol=1e-5), (
        f"Scorer mismatch for {model_name}: official={official}, direct={direct}"
    )
    print(f"{model_name}: official-predict scorer check passed")

print("All scorers ready:", list(score_functions))

LightFM_identity: official-predict scorer check passed
LightFM_content_v1: official-predict scorer check passed
All scorers ready: ['ALS_only', 'LightFM_identity', 'LightFM_content_v1']


In [9]:
def popularity_fallback_scores(user_id):
    scores = popularity_scores.copy()
    scores[seen_train[user_id].indices] = -np.inf
    return scores


def top_k_from_scores(scores, user_id, model_name, tie_seed, k=K):
    row = np.asarray(scores, dtype=np.float64).copy()

    # Users with no positive train interactions receive the agreed fallback list.
    if not u_has_pos[user_id]:
        row = popularity_fallback_scores(user_id)
    elif model_name == "ALS_only":
        row[~item_has_als_factors] = -np.inf

    # Never recommend any movie that appeared in the user's training history.
    row[seen_train[user_id].indices] = -np.inf
    eligible = np.isfinite(row)
    if int(eligible.sum()) < k:
        raise RuntimeError(f"Only {int(eligible.sum())} eligible movies for user {user_id}")

    # Seeded per-user tie-break. The same noise vector is used across models for a given
    # user/seed; epsilon is tiny relative to the model score scale.
    tie_rng = np.random.default_rng(np.random.SeedSequence([SEED, int(tie_seed), int(user_id)]))
    tie_noise = tie_rng.random(n_items)
    scale = max(1.0, float(np.max(np.abs(row[eligible]))))
    adjusted = row.copy()
    adjusted[eligible] += tie_noise[eligible] * (1e-8 * scale)

    candidates = np.argpartition(adjusted, -k)[-k:]
    return candidates[np.argsort(adjusted[candidates])[::-1]].astype(np.int32)

seen_sets = {int(u): set(seen_train[int(u)].indices.tolist()) for u in eval_users}
recommendations = {}
start_all = time.time()

for model_name, score_fn in score_functions.items():
    start_model = time.time()
    recs_by_seed = {seed: {} for seed in range(N_TIE_SEEDS)}
    print(f"\nScoring and ranking {model_name} ...")

    for start in range(0, len(eval_users), SCORE_BATCH_SIZE):
        user_batch = eval_users[start:start + SCORE_BATCH_SIZE]
        batch_scores = score_fn(user_batch)
        if batch_scores.shape != (len(user_batch), n_items):
            raise ValueError(
                f"{model_name} returned shape {batch_scores.shape}; "
                f"expected {(len(user_batch), n_items)}"
            )
        if not np.isfinite(batch_scores).all():
            raise ValueError(f"Non-finite scores from {model_name}")

        for row_idx, user_id in enumerate(user_batch):
            raw_scores = batch_scores[row_idx]
            for tie_seed in range(N_TIE_SEEDS):
                recs_by_seed[tie_seed][int(user_id)] = top_k_from_scores(
                    raw_scores, int(user_id), model_name, tie_seed
                )

        processed = min(start + len(user_batch), len(eval_users))
        if processed == len(user_batch) or processed % 512 < SCORE_BATCH_SIZE or processed == len(eval_users):
            print(f"  ranked {processed:,}/{len(eval_users):,} users")

    recommendations[model_name] = recs_by_seed
    print(f"{model_name} completed in {(time.time() - start_model) / 60:.1f} min")
    gc.collect()

print(f"\nAll ranking completed in {(time.time() - start_all) / 60:.1f} min")

# Recommendation-list integrity checks, including no seen items or duplicates.
for model_name, by_seed in recommendations.items():
    for tie_seed, user_recs in by_seed.items():
        assert len(user_recs) == len(eval_users)
        for user_id, slate in user_recs.items():
            assert len(slate) == K and len(set(map(int, slate))) == K
            assert seen_sets[user_id].isdisjoint(map(int, slate)), (model_name, tie_seed, user_id)
    print(f"{model_name}: slate integrity checks passed across {N_TIE_SEEDS} seeds")


Scoring and ranking ALS_only ...
  ranked 64/5,661 users
  ranked 512/5,661 users
  ranked 1,024/5,661 users
  ranked 1,536/5,661 users
  ranked 2,048/5,661 users
  ranked 2,560/5,661 users
  ranked 3,072/5,661 users
  ranked 3,584/5,661 users
  ranked 4,096/5,661 users
  ranked 4,608/5,661 users
  ranked 5,120/5,661 users
  ranked 5,632/5,661 users
  ranked 5,661/5,661 users
ALS_only completed in 0.4 min

Scoring and ranking LightFM_identity ...
  ranked 64/5,661 users
  ranked 512/5,661 users
  ranked 1,024/5,661 users
  ranked 1,536/5,661 users
  ranked 2,048/5,661 users
  ranked 2,560/5,661 users
  ranked 3,072/5,661 users
  ranked 3,584/5,661 users
  ranked 4,096/5,661 users
  ranked 4,608/5,661 users
  ranked 5,120/5,661 users
  ranked 5,632/5,661 users
  ranked 5,661/5,661 users
LightFM_identity completed in 0.4 min

Scoring and ranking LightFM_content_v1 ...
  ranked 64/5,661 users
  ranked 512/5,661 users
  ranked 1,024/5,661 users
  ranked 1,536/5,661 users
  ranked 2,048/5,

## Validation metrics

Report overall, tail, strict-cold and warm-tail performance at K=10.


In [10]:
discounts = 1.0 / np.log2(np.arange(2, K + 2, dtype=np.float64))


def metrics_for_slate(slate, relevant_items):
    relevant_items = set(relevant_items)
    hits = np.asarray([int(item) in relevant_items for item in slate[:K]], dtype=np.float64)
    recall = float(hits.sum() / len(relevant_items)) if relevant_items else 0.0
    idcg_len = min(len(relevant_items), K)
    idcg = float(discounts[:idcg_len].sum()) if idcg_len else 0.0
    ndcg = float((hits * discounts[:len(hits)]).sum() / idcg) if idcg > 0 else 0.0
    return {
        "ndcg@10": ndcg,
        "recall@10": recall,
        "hit_rate@10": float(hits.sum() > 0),
        "hit_count@10": float(hits.sum()),
    }


def ranking_metrics_at_10(recommendations_by_user, relevant_by_user):
    """Return per-user top-10 metrics for a dictionary of recommendation slates."""
    rows = {
        int(user_id): metrics_for_slate(
            slate, relevant_by_user.get(int(user_id), set())
        )
        for user_id, slate in recommendations_by_user.items()
    }
    frame = pd.DataFrame.from_dict(rows, orient="index")
    frame.index.name = "u"
    return frame


user_metrics = {model_name: {} for model_name in recommendations}
absolute_metric_rows = []

for scope_name, rel_dict in relevant_by_scope.items():
    users_scope = np.asarray(sorted(rel_dict), dtype=np.int64)
    print(f"Evaluating {scope_name}: {len(users_scope):,} users")

    for model_name in recommendations:
        per_seed_frames = []
        for tie_seed in range(N_TIE_SEEDS):
            values = []
            recs = recommendations[model_name][tie_seed]
            for user_id in users_scope:
                values.append(metrics_for_slate(recs[int(user_id)], rel_dict[int(user_id)]))
            frame = pd.DataFrame(values, index=users_scope)
            frame.index.name = "u"
            per_seed_frames.append(frame)

        averaged = sum(per_seed_frames) / float(N_TIE_SEEDS)
        averaged.index.name = "u"
        user_metrics[model_name][scope_name] = averaged

        for metric in ("ndcg@10", "recall@10", "hit_rate@10", "hit_count@10"):
            absolute_metric_rows.append({
                "model": model_name,
                "scope": scope_name,
                "metric": metric,
                "users": len(users_scope),
                "mean": float(averaged[metric].mean()),
            })

absolute_metrics = pd.DataFrame(absolute_metric_rows)
print("\nVALIDATION METRIC POINT ESTIMATES")
display(
    absolute_metrics.loc[absolute_metrics["metric"].isin(["ndcg@10", "recall@10", "hit_rate@10"])]
    .pivot_table(index=["scope", "metric"], columns="model", values="mean")
    .round(6)
)

# Sanity check: every model has the same users in every scope.
for scope_name, rel_dict in relevant_by_scope.items():
    for model_name in user_metrics:
        assert user_metrics[model_name][scope_name].index.tolist() == sorted(rel_dict)

Evaluating overall: 5,661 users
Evaluating tail: 948 users
Evaluating strict_cold: 50 users
Evaluating warm_tail: 931 users

VALIDATION METRIC POINT ESTIMATES


model                    ALS_only  LightFM_content_v1  LightFM_identity
scope       metric                                                     
overall     hit_rate@10  0.302067            0.251546          0.255255
            ndcg@10      0.081462            0.062729          0.063860
            recall@10    0.106220            0.082748          0.082599
strict_cold hit_rate@10  0.000000            0.000000          0.000000
            ndcg@10      0.000000            0.000000          0.000000
            recall@10    0.000000            0.000000          0.000000
tail        hit_rate@10  0.000000            0.000000          0.000000
            ndcg@10      0.000000            0.000000          0.000000
            recall@10    0.000000            0.000000          0.000000
warm_tail   hit_rate@10  0.000000            0.000000          0.000000
            ndcg@10      0.000000            0.000000          0.000000
            recall@10    0.000000            0.000000          0.000000

## Paired uncertainty and model guardrail

Bootstrap paired user-level differences. A candidate must retain at least 98% of ALS overall NDCG@10 before it can replace the ALS reference.


In [11]:
def paired_bootstrap(delta_values, draw_indices):
    delta_values = np.asarray(delta_values, dtype=np.float64)
    point = float(delta_values.mean())
    boot_means = delta_values[draw_indices].mean(axis=1)
    low, high = np.percentile(boot_means, [2.5, 97.5])
    return point, float(low), float(high)

bootstrap_rng = np.random.default_rng(SEED)
bootstrap_indices = {}
for scope_name, rel_dict in relevant_by_scope.items():
    n_scope = len(rel_dict)
    bootstrap_indices[scope_name] = bootstrap_rng.integers(
        0, n_scope, size=(N_BOOT, n_scope), endpoint=False
    )

comparisons = [
    ("ALS_only", "LightFM_identity"),
    ("ALS_only", "LightFM_content_v1"),
    ("LightFM_identity", "LightFM_content_v1"),
]
metrics_to_compare = ("ndcg@10", "recall@10", "hit_rate@10")
paired_rows = []

for scope_name, rel_dict in relevant_by_scope.items():
    users_scope = np.asarray(sorted(rel_dict), dtype=np.int64)
    draws = bootstrap_indices[scope_name]

    for reference_name, candidate_name in comparisons:
        reference_frame = user_metrics[reference_name][scope_name].loc[users_scope]
        candidate_frame = user_metrics[candidate_name][scope_name].loc[users_scope]

        for metric in metrics_to_compare:
            reference_values = reference_frame[metric].to_numpy(dtype=np.float64)
            candidate_values = candidate_frame[metric].to_numpy(dtype=np.float64)
            delta = candidate_values - reference_values
            point, low, high = paired_bootstrap(delta, draws)
            paired_rows.append({
                "scope": scope_name,
                "reference": reference_name,
                "candidate": candidate_name,
                "metric": metric,
                "users": len(users_scope),
                "reference_mean": float(reference_values.mean()),
                "candidate_mean": float(candidate_values.mean()),
                "paired_delta": point,
                "delta_ci95_low": low,
                "delta_ci95_high": high,
            })

paired_ci_results = pd.DataFrame(paired_rows)
print("PAIRED VALIDATION COMPARISONS")
display(
    paired_ci_results.loc[
        paired_ci_results["metric"].isin(["ndcg@10", "recall@10"])
    ].round(6)
)

primary = paired_ci_results.loc[
    (paired_ci_results["scope"] == "tail")
    & (paired_ci_results["metric"] == "ndcg@10")
    & (paired_ci_results["reference"] == "LightFM_identity")
]
print("\nPRIMARY QUESTION: does item metadata improve LightFM tail View A NDCG@10?")
display(primary.round(6))


PAIRED VALIDATION COMPARISONS


,scope,reference,candidate,metric,users,reference_mean,candidate_mean,paired_delta,delta_ci95_low,delta_ci95_high
0,overall,ALS_only,LightFM_identity,ndcg@10,5661,0.081462,0.063860,-0.017602,-0.021017,-0.014213
1,overall,ALS_only,LightFM_identity,recall@10,5661,0.106220,0.082599,-0.023621,-0.028082,-0.018911
3,overall,ALS_only,LightFM_content_v1,ndcg@10,5661,0.081462,0.062729,-0.018733,-0.022402,-0.015505
4,overall,ALS_only,LightFM_content_v1,recall@10,5661,0.106220,0.082748,-0.023472,-0.028381,-0.018753
6,overall,LightFM_identity,LightFM_content_v1,ndcg@10,5661,0.063860,0.062729,-0.001132,-0.003558,0.001309
7,overall,LightFM_identity,LightFM_content_v1,recall@10,5661,0.082599,0.082748,0.000149,-0.003417,0.003487
9,tail,ALS_only,LightFM_identity,ndcg@10,948,0.000000,0.000000,0.000000,0.000000,0.000000
10,tail,ALS_only,LightFM_identity,recall@10,948,0.000000,0.000000,0.000000,0.000000,0.000000
12,tail,ALS_only,LightFM_content_v1,ndcg@10,948,0.000000,0.000000,0.000000,0.000000,0.000000
13,tail,ALS_only,LightFM_content_v1,recall@10,948,0.000000,0.000000,0.000000,0.000000,0.000000



PRIMARY QUESTION: does item metadata improve LightFM tail View A NDCG@10?


,scope,reference,candidate,metric,users,reference_mean,candidate_mean,paired_delta,delta_ci95_low,delta_ci95_high
15,tail,LightFM_identity,LightFM_content_v1,ndcg@10,948,0.0,0.0,0.0,0.0,0.0


In [12]:
als_overall_frame = user_metrics["ALS_only"]["overall"]
als_overall_ndcg_values = als_overall_frame["ndcg@10"].to_numpy(dtype=np.float64)
als_overall_ndcg = float(als_overall_ndcg_values.mean())
retention_floor = RETENTION_FRACTION * als_overall_ndcg

# Build a concise table with overall, tail, cold and warm-tail point estimates.
decision_rows = []
for model_name in ("ALS_only", "LightFM_identity", "LightFM_content_v1"):
    overall = user_metrics[model_name]["overall"]
    tail = user_metrics[model_name]["tail"]
    cold = user_metrics[model_name]["strict_cold"]
    warm = user_metrics[model_name]["warm_tail"]

    overall_ndcg = float(overall["ndcg@10"].mean())
    overall_recall = float(overall["recall@10"].mean())
    guard_low, guard_high = np.nan, np.nan
    guard_point = np.nan
    guard_pass = True if model_name == "ALS_only" else bool(overall_ndcg >= retention_floor)

    if model_name != "ALS_only":
        candidate_overall_values = overall["ndcg@10"].to_numpy(dtype=np.float64)
        guard_margin = candidate_overall_values - RETENTION_FRACTION * als_overall_ndcg_values
        guard_point, guard_low, guard_high = paired_bootstrap(
            guard_margin, bootstrap_indices["overall"]
        )

    decision_rows.append({
        "model": model_name,
        "overall_users": len(overall),
        "overall_ndcg@10": overall_ndcg,
        "overall_recall@10": overall_recall,
        "tail_users": len(tail),
        "tail_ndcg@10": float(tail["ndcg@10"].mean()),
        "tail_recall@10": float(tail["recall@10"].mean()),
        "strict_cold_users": len(cold),
        "strict_cold_ndcg@10": float(cold["ndcg@10"].mean()),
        "strict_cold_recall@10": float(cold["recall@10"].mean()),
        "warm_tail_users": len(warm),
        "warm_tail_ndcg@10": float(warm["ndcg@10"].mean()),
        "warm_tail_recall@10": float(warm["recall@10"].mean()),
        "als_overall_ndcg@10": als_overall_ndcg,
        "overall_retention_floor": retention_floor,
        "overall_guardrail_pass_point_estimate": guard_pass,
        "guardrail_margin_G": guard_point,
        "guardrail_G_ci95_low": guard_low,
        "guardrail_G_ci95_high": guard_high,
    })

decision_table = pd.DataFrame(decision_rows).sort_values(
    ["overall_guardrail_pass_point_estimate", "tail_ndcg@10"],
    ascending=[False, False],
).reset_index(drop=True)

print("CANDIDATE DECISION TABLE")
display(decision_table.round(6))

# A small config artifact for reproducibility.
run_config = {
    "notebook": "05_final_hybrid_evaluation.ipynb",
    "dataset_subset": NAME,
    "positive_rating_threshold": REL,
    "validation_window_start_utc": "2016-01-01T00:00:00Z",
    "tail_definition": "fewer than 20 positive train interactions",
    "strict_cold_definition": "zero positive train interactions",
    "warm_tail_definition": "1-19 positive train interactions",
    "test_split_used": False,
    "lightfm": {
        "loss": "warp",
        "components": LIGHTFM_COMPONENTS,
        "epochs": LIGHTFM_EPOCHS,
        "learning_rate": LIGHTFM_LEARNING_RATE,
        "item_alpha": LIGHTFM_ITEM_ALPHA,
        "user_alpha": LIGHTFM_USER_ALPHA,
        "max_sampled": LIGHTFM_MAX_SAMPLED,
        "num_threads": NUM_THREADS,
        "metadata_weighting": "for rows with metadata: identity feature mass 0.5, normalized metadata feature mass 0.5; empty metadata rows use identity mass 1.0",
    },
    "bootstrap_resamples": N_BOOT,
    "tie_breaking_seeds": N_TIE_SEEDS,
    "random_seed": SEED,
    "als_overall_view_a_ndcg_at_10": als_overall_ndcg,
    "retention_fraction": RETENTION_FRACTION,
    "retention_floor": retention_floor,
}

REPORT_DIR = _PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save per-model metrics with absolute user-bootstrap CIs.
absolute_rows = []
for scope_name, rel_dict in relevant_by_scope.items():
    users_scope = np.asarray(sorted(rel_dict), dtype=np.int64)
    draws = bootstrap_indices[scope_name]
    for model_name in user_metrics:
        frame = user_metrics[model_name][scope_name].loc[users_scope]
        for metric in metrics_to_compare:
            values = frame[metric].to_numpy(dtype=np.float64)
            point, low, high = paired_bootstrap(values, draws)
            absolute_rows.append({
                "model": model_name,
                "scope": scope_name,
                "metric": metric,
                "users": len(users_scope),
                "mean": point,
                "ci95_low": low,
                "ci95_high": high,
            })
absolute_ci_table = pd.DataFrame(absolute_rows)

metrics_path = REPORT_DIR / "lightfm_validation_metrics.csv"
paired_path = REPORT_DIR / "lightfm_paired_validation_ci.csv"
decision_path = REPORT_DIR / "lightfm_candidate_decision.csv"
config_path = REPORT_DIR / "lightfm_run_config.json"
user_metrics_path = REPORT_DIR / "lightfm_user_metrics.csv"

absolute_ci_table.to_csv(metrics_path, index=False)
paired_ci_results.to_csv(paired_path, index=False)
decision_table.to_csv(decision_path, index=False)
(Path(config_path)).write_text(json.dumps(run_config, indent=2) + "\n", encoding="utf-8")

user_metric_export_rows = []
for model_name, scopes in user_metrics.items():
    for scope_name, frame in scopes.items():
        export = frame.reset_index().copy()
        export.insert(0, "scope", scope_name)
        export.insert(0, "model", model_name)
        user_metric_export_rows.append(export)
pd.concat(user_metric_export_rows, ignore_index=True).to_csv(user_metrics_path, index=False)

print("\nSaved reports:")
for path in (metrics_path, paired_path, decision_path, config_path, user_metrics_path):
    print(path)

print("\nOVERALL GUARDRAIL")
print(f"ALS overall View A NDCG@10: {als_overall_ndcg:.6f}")
print(f"98% retention floor:        {retention_floor:.6f}")
eligible = decision_table.loc[
    (decision_table["model"] != "ALS_only")
    & decision_table["overall_guardrail_pass_point_estimate"]
]
if len(eligible):
    selected = eligible.sort_values("tail_ndcg@10", ascending=False).iloc[0]
    print("Best LightFM candidate passing point-estimate guardrail:", selected["model"])
    print(f"Overall NDCG@10: {selected['overall_ndcg@10']:.6f} | Tail NDCG@10: {selected['tail_ndcg@10']:.6f}")
    print(
        "Guardrail G 95% CI: "
        f"[{selected['guardrail_G_ci95_low']:+.6f}, {selected['guardrail_G_ci95_high']:+.6f}]"
    )
else:
    print("No LightFM candidate passes the overall point-estimate guardrail. Keep ALS as the project reference.")

print("Test split used: No")


CANDIDATE DECISION TABLE


,model,overall_users,overall_ndcg@10,overall_recall@10,tail_users,tail_ndcg@10,tail_recall@10,strict_cold_users,strict_cold_ndcg@10,strict_cold_recall@10,warm_tail_users,warm_tail_ndcg@10,warm_tail_recall@10,als_overall_ndcg@10,overall_retention_floor,overall_guardrail_pass_point_estimate,guardrail_margin_G,guardrail_G_ci95_low,guardrail_G_ci95_high
0,ALS_only,5661,0.081462,0.106220,948,0.0,0.0,50,0.0,0.0,931,0.0,0.0,0.081462,0.079833,True,NaN,NaN,NaN
1,LightFM_identity,5661,0.063860,0.082599,948,0.0,0.0,50,0.0,0.0,931,0.0,0.0,0.081462,0.079833,False,-0.015973,-0.019339,-0.012606
2,LightFM_content_v1,5661,0.062729,0.082748,948,0.0,0.0,50,0.0,0.0,931,0.0,0.0,0.081462,0.079833,False,-0.017104,-0.020709,-0.013898



Saved reports:
/mnt/c/Users/Lenovo/Documents/movie-recommender/reports/lightfm_validation_metrics.csv
/mnt/c/Users/Lenovo/Documents/movie-recommender/reports/lightfm_paired_validation_ci.csv
/mnt/c/Users/Lenovo/Documents/movie-recommender/reports/lightfm_candidate_decision.csv
/mnt/c/Users/Lenovo/Documents/movie-recommender/reports/lightfm_run_config.json
/mnt/c/Users/Lenovo/Documents/movie-recommender/reports/lightfm_user_metrics.csv

OVERALL GUARDRAIL
ALS overall View A NDCG@10: 0.081462
98% retention floor:        0.079833
No LightFM candidate passes the overall point-estimate guardrail. Keep ALS as the project reference.
Test split used: No


## Selective warm-tail reranking

A candidate is selected using LightFM Content-v1. Its rank in the eligible full catalogue determines whether it can replace the last ALS recommendation. The rank gate is chosen on validation; it is not chosen using test results.


In [13]:

# Selective, rank-gated warm-tail reranking.
# Does not use validation relevance to select candidates.
# Candidate quality thresholds are exploratory and require independent
# evaluation before making a final claim.

TAIL_MODEL = "LightFM_content_v1"
gate_values = [0, 10, 25, 50, 100, 250, 500, 1000, 2000, 5000, 17241]

warm_users = set(relevant_by_scope["warm_tail"].keys())
tail_candidate_ids = np.flatnonzero(
    np.asarray(warm_tail_mask, dtype=bool)
    & np.asarray(item_has_als_factors, dtype=bool)
)

# Step A: identify each user's best warm-tail candidate outside their
# existing ALS top 10, and its rank in Content-v1's eligible full catalogue.
candidate_meta = {}

for user_id in eval_users:
    row = np.asarray(
        score_functions[TAIL_MODEL]([user_id])
    )[0].astype(np.float64, copy=True)

    # Match the existing fallback policy for users without positive history.
    if not u_has_pos[user_id]:
        row = np.asarray(
            popularity_fallback_scores(user_id),
            dtype=np.float64
        ).copy()

    row[seen_train[user_id].indices] = -np.inf

    valid_tail = tail_candidate_ids[np.isfinite(row[tail_candidate_ids])]

    if len(valid_tail):
        # We only need the top 11 tail candidates: ALS already contains
        # at most 10 movies, so one of these must be outside the ALS top 10.
        n_top = min(11, len(valid_tail))
        tail_scores = row[valid_tail]
        top_idx = np.argpartition(tail_scores, -n_top)[-n_top:]
        top_order = valid_tail[
            top_idx[np.argsort(tail_scores[top_idx])[::-1]]
        ]
    else:
        top_order = np.array([], dtype=int)

    for seed in range(10):
        base = np.asarray(
            recommendations["ALS_only"][seed][user_id],
            dtype=np.int32
        )
        base_set = set(map(int, base))

        candidate = next(
            (int(item) for item in top_order if int(item) not in base_set),
            None
        )

        if candidate is None:
            candidate_rank = None
        else:
            candidate_score = row[candidate]
            candidate_rank = 1 + int(
                np.sum(np.isfinite(row) & (row > candidate_score))
            )

        candidate_meta[(seed, user_id)] = (candidate, candidate_rank)


# Step B: evaluate each rank gate over the same 10 ALS recommendation seeds.
gate_rows = []

for gate in gate_values:
    for seed in range(10):
        gated_recs = {}
        injected_users = set()

        for user_id in eval_users:
            base = np.asarray(
                recommendations["ALS_only"][seed][user_id],
                dtype=np.int32
            )

            candidate, candidate_rank = candidate_meta[(seed, user_id)]

            if (
                gate > 0
                and candidate is not None
                and candidate_rank is not None
                and candidate_rank <= gate
            ):
                gated_recs[user_id] = np.concatenate(
                    [base[:-1], np.array([candidate], dtype=np.int32)]
                )
                injected_users.add(user_id)
            else:
                gated_recs[user_id] = base.copy()

        overall_m = ranking_metrics_at_10(gated_recs, overall_relevant)
        tail_m = ranking_metrics_at_10(gated_recs, relevant_by_scope["warm_tail"])

        for user_id in eval_users:
            gate_rows.append({
                "max_content_rank": gate,
                "seed": seed,
                "user_id": user_id,
                "overall_ndcg": overall_m.loc[user_id, "ndcg@10"],
                "overall_recall": overall_m.loc[user_id, "recall@10"],
                "overall_hit_rate": overall_m.loc[user_id, "hit_rate@10"],
                "warm_tail_recall": (
                    tail_m.loc[user_id, "recall@10"]
                    if user_id in warm_users else np.nan
                ),
                "warm_tail_hit_rate": (
                    tail_m.loc[user_id, "hit_rate@10"]
                    if user_id in warm_users else np.nan
                ),
                "injected": user_id in injected_users,
            })

gate_results = pd.DataFrame(gate_rows)

# Average each user's outcome across the 10 seeds before summarizing users.
per_user_gate = (
    gate_results
    .groupby(["max_content_rank", "user_id"], as_index=False)
    .agg(
        overall_ndcg=("overall_ndcg", "mean"),
        overall_recall=("overall_recall", "mean"),
        overall_hit_rate=("overall_hit_rate", "mean"),
        warm_tail_recall=("warm_tail_recall", "mean"),
        warm_tail_hit_rate=("warm_tail_hit_rate", "mean"),
        injection_frequency=("injected", "mean"),
    )
)

summary_rows = []

for gate in gate_values:
    group = per_user_gate[per_user_gate["max_content_rank"] == gate]
    tail_group = group[group["user_id"].isin(warm_users)]

    summary_rows.append({
        "max_content_rank": gate,
        "overall_ndcg": group["overall_ndcg"].mean(),
        "overall_recall": group["overall_recall"].mean(),
        "overall_hit_rate": group["overall_hit_rate"].mean(),
        "warm_tail_recall": tail_group["warm_tail_recall"].mean(),
        "warm_tail_hit_rate": tail_group["warm_tail_hit_rate"].mean(),
        "users_injected_any_seed": int(
            (group["injection_frequency"] > 0).sum()
        ),
        "user_seed_injection_rate": group["injection_frequency"].mean(),
    })

gate_summary = pd.DataFrame(summary_rows).set_index("max_content_rank")

baseline_ndcg = gate_summary.loc[0, "overall_ndcg"]
baseline_tail_recall = gate_summary.loc[0, "warm_tail_recall"]
baseline_tail_hit = gate_summary.loc[0, "warm_tail_hit_rate"]

gate_summary["overall_ndcg_loss_pct"] = (
    (baseline_ndcg - gate_summary["overall_ndcg"])
    / baseline_ndcg * 100
)
gate_summary["warm_tail_recall_gain"] = (
    gate_summary["warm_tail_recall"] - baseline_tail_recall
)
gate_summary["warm_tail_hit_rate_gain"] = (
    gate_summary["warm_tail_hit_rate"] - baseline_tail_hit
)

display(gate_summary.round(6))


,overall_ndcg,overall_recall,overall_hit_rate,warm_tail_recall,warm_tail_hit_rate,users_injected_any_seed,user_seed_injection_rate,overall_ndcg_loss_pct,warm_tail_recall_gain,warm_tail_hit_rate_gain
max_content_rank,,,,,,,,,,
0,0.081462,0.106220,0.302067,0.000000,0.000000,0,0.000000,0.000000,0.000000,0.000000
10,0.081448,0.106194,0.301890,0.000000,0.000000,19,0.003356,0.017230,0.000000,0.000000
25,0.081448,0.106194,0.301890,0.000000,0.000000,36,0.006359,0.017230,0.000000,0.000000
50,0.081448,0.106194,0.301890,0.000000,0.000000,74,0.013072,0.017230,0.000000,0.000000
100,0.081407,0.106093,0.301537,0.000153,0.001074,155,0.027380,0.067400,0.000153,0.001074
250,0.081283,0.105807,0.301007,0.001586,0.003222,498,0.087970,0.219694,0.001586,0.003222
500,0.081007,0.105221,0.300300,0.004629,0.008593,1273,0.224872,0.558150,0.004629,0.008593
1000,0.080264,0.103449,0.296767,0.007672,0.012889,3055,0.539657,1.471136,0.007672,0.012889
2000,0.078607,0.099668,0.289525,0.008746,0.013963,5473,0.966790,3.504927,0.008746,0.013963


In [14]:

# Paired bootstrap CIs for selective reranking gates.
# User-level metrics are already averaged across the 10 tie-break seeds.

rng_ci = np.random.default_rng(2027)
N_BOOT = 3000
BATCH_SIZE = 100

gates_to_test = [100, 250, 500, 1000, 2000]

all_users = sorted(overall_relevant.keys())
warm_users = sorted(relevant_by_scope["warm_tail"].keys())

user_tables = {
    gate: (
        per_user_gate[
            per_user_gate["max_content_rank"] == gate
        ]
        .set_index("user_id")
        .reindex(all_users)
    )
    for gate in [0] + gates_to_test
}

def paired_bootstrap_ci(differences):
    differences = np.asarray(differences, dtype=float)
    n = len(differences)
    boot_means = np.empty(N_BOOT)

    for start in range(0, N_BOOT, BATCH_SIZE):
        batch_n = min(BATCH_SIZE, N_BOOT - start)
        indices = rng_ci.integers(0, n, size=(batch_n, n))
        boot_means[start:start + batch_n] = (
            differences[indices].mean(axis=1)
        )

    return (
        float(differences.mean()),
        float(np.quantile(boot_means, 0.025)),
        float(np.quantile(boot_means, 0.975)),
    )

baseline = user_tables[0]
ci_rows = []

comparisons = [
    (
        "overall_ndcg",
        "Overall NDCG@10 change",
        all_users,
    ),
    (
        "warm_tail_recall",
        "Warm-tail Recall@10 gain",
        warm_users,
    ),
    (
        "warm_tail_hit_rate",
        "Warm-tail HitRate@10 gain",
        warm_users,
    ),
]

for gate in gates_to_test:
    current = user_tables[gate]

    for col, label, users in comparisons:
        delta = (
            current.loc[users, col].to_numpy(dtype=float)
            - baseline.loc[users, col].to_numpy(dtype=float)
        )

        mean, low, high = paired_bootstrap_ci(delta)

        ci_rows.append({
            "gate": gate,
            "metric": label,
            "mean_change": mean,
            "ci95_low": low,
            "ci95_high": high,
            "users": len(users),
        })

selective_ci = pd.DataFrame(ci_rows)
display(selective_ci.round(6))


,gate,metric,mean_change,ci95_low,ci95_high,users
0,100,Overall NDCG@10 change,-0.000055,-0.000142,0.000013,5661
1,100,Warm-tail Recall@10 gain,0.000153,0.000000,0.000460,931
2,100,Warm-tail HitRate@10 gain,0.001074,0.000000,0.003222,931
3,250,Overall NDCG@10 change,-0.000179,-0.000328,-0.000052,5661
4,250,Warm-tail Recall@10 gain,0.001586,0.000000,0.003941,931
5,250,Warm-tail HitRate@10 gain,0.003222,0.000000,0.007519,931
6,500,Overall NDCG@10 change,-0.000455,-0.000732,-0.000199,5661
7,500,Warm-tail Recall@10 gain,0.004629,0.001432,0.008593,931
8,500,Warm-tail HitRate@10 gain,0.008593,0.003222,0.015038,931
9,1000,Overall NDCG@10 change,-0.001198,-0.001604,-0.000827,5661


The validation results support a 500-rank gate as a compromise: it recovers warm-tail hits while keeping the overall NDCG change small. The gate is frozen before the final test evaluation.


## Refit on train + validation

Refit ALS and LightFM Content-v1 using all pre-test interactions. Construct the warm-tail masks from train + validation, since those are the interactions available immediately before test.


In [15]:
# Refit on all data available before the test period.
train_val = pd.concat([train, val], ignore_index=True)

positive_tv = (
    train_val.loc[train_val["rating"] >= REL, ["u", "i"]]
    .drop_duplicates(["u", "i"])
)
interactions_tv = sp.coo_matrix(
    (
        np.ones(len(positive_tv), dtype=np.float32),
        (
            positive_tv["u"].to_numpy(dtype=np.int32),
            positive_tv["i"].to_numpy(dtype=np.int32),
        ),
    ),
    shape=(n_users, n_items),
)

seen_pairs_tv = train_val[["u", "i"]].drop_duplicates()
seen_tv = sp.csr_matrix(
    (
        np.ones(len(seen_pairs_tv), dtype=np.float32),
        (
            seen_pairs_tv["u"].to_numpy(dtype=np.int32),
            seen_pairs_tv["i"].to_numpy(dtype=np.int32),
        ),
    ),
    shape=(n_users, n_items),
)
seen_tv.sum_duplicates()
seen_tv.data[:] = 1.0

n_pos_tv = np.bincount(
    positive_tv["i"].to_numpy(dtype=np.int64), minlength=n_items
)
tail_mask_tv = n_pos_tv <= 19
warm_tail_mask_tv = (n_pos_tv >= 1) & (n_pos_tv <= 19)
strict_cold_mask_tv = n_pos_tv == 0
item_has_als_factors_tv = n_pos_tv > 0
u_has_pos_tv = np.zeros(n_users, dtype=bool)
u_has_pos_tv[positive_tv["u"].unique()] = True
popularity_scores_tv = np.bincount(
    train_val["i"].to_numpy(dtype=np.int64), minlength=n_items
).astype(np.float64)

assert interactions_tv.shape == (n_users, n_items)
assert item_features_v1.shape[0] == n_items
assert seen_tv.shape == (n_users, n_items)
assert np.array_equal(item_map["i"].to_numpy(), np.arange(n_items))

print("Pre-test training rows:", f"{len(train_val):,}")
print("Positive user-item pairs:", f"{len(positive_tv):,}")
print("Warm-tail / strict-cold catalogue items:",
      int(warm_tail_mask_tv.sum()), "/", int(strict_cold_mask_tv.sum()))


Pre-test training rows: 3,056,877
Positive user-item pairs: 1,521,298
Warm-tail / strict-cold catalogue items: 10435 / 1021


In [16]:

# Confirm the new training inputs and feature matrix.
assert len(train_val) > len(train)
assert interactions_tv.shape == (n_users, n_items)
assert item_features_v1.shape[0] == n_items
assert warm_tail_mask_tv.shape == (n_items,)
assert np.array_equal(item_map["i"].to_numpy(), np.arange(n_items))

# --------------------------------------------------
# 1. Refit implicit ALS on train + validation
# --------------------------------------------------
print("=== Fitting ALS on train + validation ===")

start = time.time()

als_final = fit_implicit_als_train_only(
    train_val,
    k=64,
    alpha=5.0,
    lam=80.0,
    n_iter=10,
    seed=42,
)

P_final = als_final["P"]
Q_final = als_final["Q"]

assert P_final.shape == (n_users, 64)
assert Q_final.shape == (n_items, 64)
assert np.isfinite(P_final).all() and np.isfinite(Q_final).all()

print(f"ALS complete in {(time.time() - start) / 60:.2f} minutes")
print("User factors:", P_final.shape)
print("Item factors:", Q_final.shape)

# --------------------------------------------------
# 2. Refit LightFM Content-v1 on train + validation
# --------------------------------------------------
print("\n=== Fitting LightFM Content-v1 on train + validation ===")

lightfm_final = LightFM(
    no_components=LIGHTFM_COMPONENTS,       # 32
    loss="warp",
    learning_rate=LIGHTFM_LEARNING_RATE,    # 0.05
    item_alpha=LIGHTFM_ITEM_ALPHA,           # 1e-6
    user_alpha=LIGHTFM_USER_ALPHA,           # 1e-6
    max_sampled=LIGHTFM_MAX_SAMPLED,         # 10
    random_state=SEED,                       # 42
)

start = time.time()

lightfm_final.fit(
    interactions_tv,
    item_features=item_features_v1,
    epochs=LIGHTFM_EPOCHS,                   # 15
    num_threads=NUM_THREADS,                 # 1
    verbose=True,
)

print(
    f"LightFM Content-v1 complete in "
    f"{(time.time() - start) / 60:.2f} minutes"
)

assert lightfm_final.user_embeddings.shape[0] == n_users
assert np.isfinite(lightfm_final.user_embeddings).all()
assert np.isfinite(lightfm_final.item_embeddings).all()

print("User embeddings:", lightfm_final.user_embeddings.shape)
print("Item-feature embeddings:", lightfm_final.item_embeddings.shape)

print("\n=== Final models ready ===")
print("ALS factors and LightFM model fitted.")
print("Fallback popularity scores:", popularity_scores_tv.shape)
print("Positive-history users:", int(u_has_pos_tv.sum()))
print("Warm-tail catalogue items:", int(warm_tail_mask_tv.sum()))
print("Strict-cold catalogue items:", int(strict_cold_mask_tv.sum()))


=== Fitting ALS on train + validation ===
ALS iteration 01/10 | elapsed 3.7s
ALS iteration 02/10 | elapsed 7.8s
ALS iteration 04/10 | elapsed 16.3s
ALS iteration 06/10 | elapsed 24.9s
ALS iteration 08/10 | elapsed 33.7s
ALS iteration 10/10 | elapsed 43.0s
ALS complete in 0.72 minutes
User factors: (20093, 64)
Item factors: (17241, 64)

=== Fitting LightFM Content-v1 on train + validation ===
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
LightFM Content-v1 complete in 0.71 minutes
User embeddings: (20093, 32)
Item-feature embeddings: (17272, 32)

=== Final models ready ===
ALS factors and LightFM model fitted.
Fallback popularity scores: (17241,)
Positive-history users: 20065
Warm-tail catalogue items: 10435
Strict-cold catalogue items: 1021


## Final test evaluation

The test set is loaded now, after model settings and the 500-rank gate have been fixed. Test recommendations exclude the user's complete pre-test history.


In [17]:
test = pd.read_parquet(DATA_DIR / "test.parquet")


# ============================================================
# 0. Freeze evaluation settings
# ============================================================
K_EVAL = 10
TIE_SEEDS = range(10)
GATES = [0, 500, 1000]  # 0 = ALS baseline
BATCH_SIZE = 64
BOOTSTRAP_REPS = 3000

# ============================================================
# 1. Construct test relevance using the locked protocol
#    rating >= 4.0 and timestamp >= 2016-01-01 UTC
# ============================================================
test_ts = pd.to_numeric(test["timestamp"], errors="raise")

test_window = test.loc[
    (test["rating"] >= REL) & (test_ts >= CUTOFF_TS),
    ["u", "i", "timestamp"],
].drop_duplicates(["u", "i"]).copy()

test_item_ids = test_window["i"].to_numpy(dtype=np.int64)

test_scope_masks = {
    "overall": np.ones(len(test_window), dtype=bool),
    "tail": tail_mask_tv[test_item_ids],
    "strict_cold": strict_cold_mask_tv[test_item_ids],
    "warm_tail": warm_tail_mask_tv[test_item_ids],
}

def build_relevant_dict(frame):
    return {
        int(user): set(group["i"].astype(int).tolist())
        for user, group in frame.groupby("u", sort=False)
    }

test_relevant = {
    scope: build_relevant_dict(test_window.loc[mask])
    for scope, mask in test_scope_masks.items()
}

test_users = sorted(test_relevant["overall"])
test_warm_users = sorted(test_relevant["warm_tail"])

assert test_users, "No positive test targets found under the locked protocol."
assert len(test_users) == len(test_relevant["overall"])
assert all(
    not (set(items) & set(seen_tv[user].indices))
    for user, items in test_relevant["overall"].items()
), "Found a test target in the user's pre-test history."

print("=== Locked test targets ===")
print("Post-cutoff positive user-movie pairs:", len(test_window))
for scope in ["overall", "tail", "warm_tail", "strict_cold"]:
    pairs = sum(len(v) for v in test_relevant[scope].values())
    print(
        f"{scope}: {pairs:,} relevant pairs across "
        f"{len(test_relevant[scope]):,} users"
    )

# ============================================================
# 2. Create scoring representations from the final models
# ============================================================
als_user_factors = P_final
als_item_factors = Q_final

lf_user_bias, lf_user_vectors = lightfm_final.get_user_representations()
lf_item_bias, lf_item_vectors = lightfm_final.get_item_representations(
    item_features_v1
)

assert als_user_factors.shape == (n_users, 64)
assert als_item_factors.shape == (n_items, 64)
assert lf_user_vectors.shape[0] == n_users
assert lf_item_vectors.shape[0] == n_items

def score_als_batch(user_ids):
    user_ids = np.asarray(user_ids, dtype=np.int64)
    return (als_user_factors[user_ids] @ als_item_factors.T).astype(
        np.float32, copy=False
    )

def score_content_batch(user_ids):
    user_ids = np.asarray(user_ids, dtype=np.int32)
    scores = lf_user_vectors[user_ids] @ lf_item_vectors.T
    scores += lf_user_bias[user_ids, None]
    scores += lf_item_bias[None, :]
    return scores.astype(np.float32, copy=False)

# Same recommendation eligibility and seeded tie-break rules used previously.
def final_top_k(scores, user_id, tie_seed):
    row = np.asarray(scores, dtype=np.float64).copy()

    if not u_has_pos_tv[user_id]:
        row = popularity_scores_tv.copy()
    else:
        row[~item_has_als_factors_tv] = -np.inf

    row[seen_tv[user_id].indices] = -np.inf
    eligible = np.isfinite(row)

    if int(eligible.sum()) < K_EVAL:
        raise RuntimeError(
            f"Only {int(eligible.sum())} eligible items for user {user_id}"
        )

    tie_rng = np.random.default_rng(
        np.random.SeedSequence([SEED, int(tie_seed), int(user_id)])
    )
    tie_noise = tie_rng.random(n_items)
    scale = max(1.0, float(np.max(np.abs(row[eligible]))))

    adjusted = row.copy()
    adjusted[eligible] += tie_noise[eligible] * (1e-8 * scale)

    candidates = np.argpartition(adjusted, -K_EVAL)[-K_EVAL:]
    return candidates[np.argsort(adjusted[candidates])[::-1]].astype(np.int32)

# ============================================================
# 3. Generate baseline and gated recommendations
# ============================================================
tail_candidate_ids = np.flatnonzero(
    warm_tail_mask_tv & item_has_als_factors_tv
)

# gate -> tie seed -> user -> recommended item IDs
test_recs = {
    gate: {seed: {} for seed in TIE_SEEDS}
    for gate in GATES
}

for start in range(0, len(test_users), BATCH_SIZE):
    batch_users = np.asarray(
        test_users[start:start + BATCH_SIZE], dtype=np.int64
    )

    als_batch = score_als_batch(batch_users)
    content_batch = score_content_batch(batch_users)

    for row_idx, user_id_value in enumerate(batch_users):
        user_id = int(user_id_value)
        als_scores = als_batch[row_idx]

        # Candidate ranking uses content-v1's full-catalogue scores,
        # excluding the user's complete pre-test history.
        content_row = np.asarray(
            content_batch[row_idx], dtype=np.float64
        ).copy()

        # Preserve the established fallback for users with no positive history.
        if not u_has_pos_tv[user_id]:
            content_row = popularity_scores_tv.copy()

        content_row[seen_tv[user_id].indices] = -np.inf

        valid_tail = tail_candidate_ids[
            np.isfinite(content_row[tail_candidate_ids])
        ]

        if len(valid_tail):
            n_top = min(K_EVAL + 1, len(valid_tail))
            tail_scores = content_row[valid_tail]
            top_idx = np.argpartition(tail_scores, -n_top)[-n_top:]
            top_order = valid_tail[
                top_idx[np.argsort(tail_scores[top_idx])[::-1]]
            ]
        else:
            top_order = np.array([], dtype=np.int64)

        for seed in TIE_SEEDS:
            base = final_top_k(als_scores, user_id, seed)
            test_recs[0][seed][user_id] = base

            base_set = set(map(int, base))

            # Pick the highest-scoring warm-tail candidate not already in ALS@10.
            candidate = next(
                (
                    int(item) for item in top_order
                    if int(item) not in base_set
                ),
                None,
            )

            if candidate is None:
                candidate_rank = None
            else:
                candidate_score = content_row[candidate]
                candidate_rank = 1 + int(
                    np.sum(
                        np.isfinite(content_row)
                        & (content_row > candidate_score)
                    )
                )

            for gate in [500, 1000]:
                if (
                    candidate is not None
                    and candidate_rank is not None
                    and candidate_rank <= gate
                ):
                    gated = np.concatenate(
                        [base[:-1], np.array([candidate], dtype=np.int32)]
                    )
                else:
                    gated = base.copy()

                test_recs[gate][seed][user_id] = gated

# ============================================================
# 4. Evaluate per-user metrics at each seed and scope
# ============================================================
DISCOUNTS = 1.0 / np.log2(np.arange(2, K_EVAL + 2))

def evaluate_one_user(rec, truth):
    rec = np.asarray(rec, dtype=int)
    truth = set(truth)

    hits = np.array([int(item in truth) for item in rec], dtype=float)
    dcg = float(np.sum(hits * DISCOUNTS))

    ideal_hits = min(len(truth), K_EVAL)
    idcg = float(np.sum(DISCOUNTS[:ideal_hits])) if ideal_hits else 0.0

    return {
        "ndcg@10": dcg / idcg if idcg else 0.0,
        "recall@10": float(hits.sum() / len(truth)) if truth else 0.0,
        "hit_rate@10": float(hits.sum() > 0),
        "hit_count@10": float(hits.sum()),
    }

metric_rows = []
exposure_rows = []

for gate in GATES:
    for seed in TIE_SEEDS:
        recs_for_seed = test_recs[gate][seed]

        # Overall metrics: users with at least one relevant test target.
        for user_id, truth in test_relevant["overall"].items():
            metrics = evaluate_one_user(recs_for_seed[user_id], truth)
            metric_rows.append({
                "gate": gate,
                "seed": seed,
                "scope": "overall",
                "user_id": user_id,
                **metrics,
            })

            exposure_rows.append({
                "gate": gate,
                "seed": seed,
                "user_id": user_id,
                "warm_tail_slot_fraction": float(
                    warm_tail_mask_tv[recs_for_seed[user_id]].mean()
                ),
            })

        # Warm-tail metrics: only users with at least one warm-tail target.
        for user_id, truth in test_relevant["warm_tail"].items():
            metrics = evaluate_one_user(recs_for_seed[user_id], truth)
            metric_rows.append({
                "gate": gate,
                "seed": seed,
                "scope": "warm_tail",
                "user_id": user_id,
                **metrics,
            })

metrics_by_seed = pd.DataFrame(metric_rows)

# Average each user's results across tie-break seeds before aggregating users.
per_user_test = (
    metrics_by_seed
    .groupby(["gate", "scope", "user_id"], as_index=False)
    [["ndcg@10", "recall@10", "hit_rate@10", "hit_count@10"]]
    .mean()
)

test_summary = (
    per_user_test
    .groupby(["gate", "scope"])
    [["ndcg@10", "recall@10", "hit_rate@10", "hit_count@10"]]
    .mean()
    .round(6)
)

exposure_by_user = (
    pd.DataFrame(exposure_rows)
    .groupby(["gate", "user_id"])["warm_tail_slot_fraction"]
    .mean()
    .groupby("gate")
    .mean()
    .rename("warm_tail_recommendation_slot_fraction")
)

print("\n=== FINAL TEST METRICS ===")
display(test_summary)

print("\n=== Warm-tail recommendation exposure ===")
display(exposure_by_user.to_frame().round(6))

# ============================================================
# 5. Paired user-bootstrap confidence intervals vs ALS baseline
# ============================================================
rng_boot = np.random.default_rng(2026)
N_BOOT = BOOTSTRAP_REPS
BOOT_BATCH = 100

def paired_bootstrap_ci(differences):
    values = np.asarray(differences, dtype=np.float64)
    n = len(values)
    boot_means = np.empty(N_BOOT, dtype=np.float64)

    for start in range(0, N_BOOT, BOOT_BATCH):
        count = min(BOOT_BATCH, N_BOOT - start)
        idx = rng_boot.integers(0, n, size=(count, n))
        boot_means[start:start + count] = values[idx].mean(axis=1)

    lo, hi = np.quantile(boot_means, [0.025, 0.975])
    return float(values.mean()), float(lo), float(hi)

ci_rows = []
ci_specs = [
    ("overall", "ndcg@10", "Overall NDCG@10 change"),
    ("overall", "recall@10", "Overall Recall@10 change"),
    ("warm_tail", "recall@10", "Warm-tail Recall@10 change"),
    ("warm_tail", "hit_rate@10", "Warm-tail HitRate@10 change"),
]

indexed = per_user_test.set_index(["gate", "scope", "user_id"])

for gate in [500, 1000]:
    for scope, col, label in ci_specs:
        baseline_users = indexed.loc[(0, scope)]
        gated_users = indexed.loc[(gate, scope)]

        common_users = baseline_users.index.intersection(gated_users.index)
        delta = (
            gated_users.loc[common_users, col].to_numpy()
            - baseline_users.loc[common_users, col].to_numpy()
        )

        mean, low, high = paired_bootstrap_ci(delta)
        ci_rows.append({
            "gate": gate,
            "metric": label,
            "mean_change": mean,
            "ci95_low": low,
            "ci95_high": high,
            "users": len(common_users),
        })

test_ci = pd.DataFrame(ci_rows)

print("\n=== PAIRED 95% USER-BOOTSTRAP CIs VS ALS ===")
display(test_ci.round(6))

print(
    "\nEvaluation complete. Test results were used only for this final assessment; "
    "do not use them to adjust the rank gate."
)


=== Locked test targets ===
Post-cutoff positive user-movie pairs: 26658
overall: 26,658 relevant pairs across 5,831 users
tail: 1,885 relevant pairs across 1,140 users
warm_tail: 1,810 relevant pairs across 1,121 users
strict_cold: 75 relevant pairs across 73 users

=== FINAL TEST METRICS ===


ndcg@10  recall@10  hit_rate@10  hit_count@10
gate scope                                                    
0    overall    0.076251   0.097556     0.302864      0.418796
     warm_tail  0.000000   0.000000     0.000000      0.000000
500  overall    0.076092   0.097151     0.302864      0.417424
     warm_tail  0.001685   0.004891     0.009813      0.009813
1000 overall    0.075069   0.094994     0.297548      0.406963
     warm_tail  0.002101   0.006230     0.011597      0.011597


=== Warm-tail recommendation exposure ===


,warm_tail_recommendation_slot_fraction
gate,
0,0.000000
500,0.022072
1000,0.054279



=== PAIRED 95% USER-BOOTSTRAP CIs VS ALS ===


,gate,metric,mean_change,ci95_low,ci95_high,users
0,500,Overall NDCG@10 change,-0.000159,-0.000399,0.000071,5831
1,500,Overall Recall@10 change,-0.000405,-0.001092,0.000199,5831
2,500,Warm-tail Recall@10 change,0.004891,0.001858,0.008727,1121
3,500,Warm-tail HitRate@10 change,0.009813,0.004460,0.016057,1121
4,1000,Overall NDCG@10 change,-0.001182,-0.001568,-0.000801,5831
5,1000,Overall Recall@10 change,-0.002563,-0.003636,-0.001535,5831
6,1000,Warm-tail Recall@10 change,0.006230,0.002661,0.010645,1121
7,1000,Warm-tail HitRate@10 change,0.011597,0.005352,0.017841,1121



Evaluation complete. Test results were used only for this final assessment; do not use them to adjust the rank gate.


In [18]:
reports = _PROJECT_ROOT / "reports"
reports.mkdir(parents=True, exist_ok=True)

test_summary.reset_index().to_csv(
    reports / "lightfm_rank_gate_test_metrics.csv", index=False
)
test_ci.to_csv(
    reports / "lightfm_rank_gate_test_paired_ci.csv", index=False
)
exposure_by_user.rename_axis("gate").reset_index().to_csv(
    reports / "lightfm_rank_gate_test_exposure.csv", index=False
)

test_config = {
    "training_data": "train + validation",
    "test_split": "sample_10pct test",
    "cutoff_utc": "2016-01-01T00:00:00Z",
    "minimum_relevant_rating": 4.0,
    "warm_tail_definition": "1-19 positive interactions in train + validation",
    "selected_gate": 500,
    "secondary_gate": 1000,
    "recommendation_k": 10,
    "tie_break_seeds": 10,
    "bootstrap_replicates": 3000,
    "bootstrap_seed": 2026,
}
(reports / "lightfm_rank_gate_test_config.json").write_text(
    json.dumps(test_config, indent=2) + "\n", encoding="utf-8"
)
print("Final rank-gate test reports exported.")


Final rank-gate test reports exported.


## Interpretation

The 500-rank gate is a selective trade-off, not a new standalone recommender. It slightly reduces overall ranking metrics while improving warm-tail recall and hit rate. The deployed application remains the ALS-only prototype.
